In [3]:
import pandas as pd
import numpy as np
from sklearn.decomposition import FactorAnalysis, PCA

# Load files
csv = pd.read_csv('C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/cats/cat_breeds.csv')
map_df = pd.read_excel('C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/cats/map.xlsx', engine='openpyxl')

# Keep only numeric columns
num = csv.select_dtypes(include=[np.number])

# Determine optimal number of factors via PCA cumulative variance
pca = PCA().fit(num.dropna())
expl = np.cumsum(pca.explained_variance_ratio_)
opt = np.argmax(expl >= 0.8) + 1   # Factors needed to reach ≥80% variance

# Perform factor analysis
fa = FactorAnalysis(n_components=opt)
factors = fa.fit_transform(num.dropna())

# Create factor loadings table
loadings = pd.DataFrame(
    fa.components_.T, 
    index=num.columns, 
    columns=[f"Factor {i+1}" for i in range(opt)]
)

# Save loadings
loadings.to_csv("factor_loadings.csv")

print("Optimal factors:", opt)
print(loadings.head())


Optimal factors: 3
                     Factor 1  Factor 2  Factor 3
min_life_expectancy  0.838269  1.669683 -0.423858
max_life_expectancy  0.564536  1.516655 -0.384997
min_weight           1.601345 -0.155968  0.044498
max_weight           4.040115 -0.425662 -0.058863
family_friendly     -0.185143 -0.138393 -0.324600


🔹 Factor 1: “Physical Size & Longevity”

Strong positive loading on weight variables
Moderate loading on life expectancy
➡️ Represents body mass + lifespan traits

🔹 Factor 2: “Health & Vitality”

Life expectancy variables load strongly
➡️ Captures health robustness across breeds

🔹 Factor 3: “Sociability & Behavior”

Negative/positive loadings on temperament scores
➡️ Indicates friendliness, playfulness, grooming needs, intelligence, etc.

In [6]:
# -*- coding: utf-8 -*-
# Cluster analysis on factor scores for cat breeds
# ------------------------------------------------

import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ----------------------
# 1) Load & prepare data
# ----------------------
df = csv

# Keep only numeric columns (factor analysis works on numeric variables)
num = df.select_dtypes(include=[np.number])

# Impute missing with median and standardize
imputer = SimpleImputer(strategy='median')
X_num = imputer.fit_transform(num)
scaler = StandardScaler()
X = scaler.fit_transform(X_num)

# -------------------------------------------------------------
# 2) Choose number of factors: PCA proxy for ≥80% variance rule
# -------------------------------------------------------------
pca = PCA().fit(X)
expl_cum = np.cumsum(pca.explained_variance_ratio_)
n_factors = int(np.argmax(expl_cum >= 0.80) + 1)

# --------------------
# 3) Factor extraction
# --------------------
fa = FactorAnalysis(n_components=n_factors, random_state=42)
F = fa.fit_transform(X)  # factor scores per sample

# Put loadings in a DataFrame (variables x factors)
loadings = pd.DataFrame(
    fa.components_.T,
    index=num.columns,
    columns=[f'F{i+1}' for i in range(n_factors)]
)

# -----------------------------------------------------------
# 4) Factor naming by coarse content groups (for interpret.)
# -----------------------------------------------------------
# Group numeric variables into interpretable families
var_groups = {
    'Size & Longevity': [c for c in num.columns
                         if 'weight' in c or 'life_expectancy' in c],
    'Sociability & Behavior': [
        'family_friendly','shedding','general_health','playfulness',
        'children_friendly','grooming','intelligence','other_pets_friendly'
    ],
}

# Any remaining numeric variables go to 'Other'
used = set(var_groups['Size & Longevity']) | set(var_groups['Sociability & Behavior'])
other_vars = [c for c in num.columns if c not in used]
if other_vars:
    var_groups['Other'] = other_vars

# Score each factor vs groups using sum of absolute loadings
factor_group_scores = {}
for fcode in loadings.columns:
    factor_group_scores[fcode] = {
        grp: float(np.abs(loadings.loc[var_groups[grp], fcode]).sum())
        for grp in var_groups
    }

# Assign each factor to its most strongly associated group
assigned_names = {}
used_group_names = set()
for fcode, grp_scores in factor_group_scores.items():
    best_grp = max(grp_scores, key=grp_scores.get)
    # in case multiple factors map to the same group, keep the base name and add suffix only for disambiguation
    name = best_grp if best_grp not in used_group_names else f"{best_grp} ({fcode})"
    assigned_names[fcode] = name
    used_group_names.add(name)

# ------------------------------------------
# 5) Prepare standardized factor scores (Fz)
# ------------------------------------------
F_scaler = StandardScaler()
Fz = F_scaler.fit_transform(F)
Fz_df = pd.DataFrame(Fz, columns=[*assigned_names.keys()])

# -------------------------------------------
# 6) Choose number of clusters via silhouette
# -------------------------------------------
best_k, best_sil = None, -1
for k in range(2, 9):  # search k=2..8
    km = KMeans(n_clusters=k, n_init=50, random_state=42)
    labels = km.fit_predict(Fz_df)
    sil = silhouette_score(Fz_df, labels)
    if sil > best_sil:
        best_k, best_sil = k, sil

# Final KMeans with the chosen k
km_final = KMeans(n_clusters=best_k, n_init=100, random_state=42)
labels_final = km_final.fit_predict(Fz_df)

# ----------------------------------------------------
# 7) Human-friendly interpretation at two base levels
# ----------------------------------------------------
# Base name = strip any "(F#)" disambiguation
def base_name(name: str) -> str:
    return name.split(' (')[0]

base_map = {f: base_name(v) for f, v in assigned_names.items()}

# Per-cluster mean z-scores for each individual factor
cluster_df = Fz_df.copy()
cluster_df['cluster'] = labels_final
cluster_z_means = cluster_df.groupby('cluster').mean()

# Collapse to base dimensions by averaging all factors that belong to the same base group
base_names = sorted(set(base_map.values()))
base_cols = {b: [f for f in Fz_df.columns if base_map[f] == b] for b in base_names}
base_summary = cluster_df.groupby('cluster').apply(
    lambda g: pd.Series({b: g[cols].mean().mean() if cols else np.nan
                         for b, cols in base_cols.items()})
)
base_summary['count'] = cluster_df.groupby('cluster').size().values

# ----------------------------------------------
# 8) Name clusters & craft 3-sentence summaries
# ----------------------------------------------

hi, lo = 0.35, -0.35  # thresholds for “high”/“low” in z-space

cluster_names, cluster_descs = {}, {}
for cidx, row in base_summary.drop(columns=['count']).iterrows():
    highs = [(b, row[b]) for b in base_names if row[b] >= hi]
    lows  = [(b, row[b]) for b in base_names if row[b] <= lo]

    # Name from top two absolute base dimensions
    top = sorted(
        [(b, abs(row[b]), np.sign(row[b])) for b in base_names if not np.isnan(row[b])],
        key=lambda x: -x[1]
    )[:2]
    name_parts = [ (b if s >= 0 else f"Lower {b}") for (b, _, s) in top ]
    cname = " / ".join(name_parts) + " Breeds"
    cluster_names[cidx] = cname

    # 3-sentence description
    bits = []
    if highs: bits.append("higher on " + ", ".join([b for b, _ in highs]))
    if lows:  bits.append("lower on "  + ", ".join([b for b, _ in lows]))
    overview = "This cluster comprises breeds that are " + (" and ".join(bits) if bits else "balanced across factors") + "."
    strengths = ("It excels in " + ", ".join([b for b, _ in highs]) + ", suggesting favorable traits on these dimensions."
                 if highs else "Its main strength is a balanced profile without pronounced extremes.")
    weaknesses = ("Potential weaknesses include relatively lower scores on " + ", ".join([b for b, _ in lows]) + "."
                  if lows else "No clear weaknesses emerge; trade‑offs are minimal across the measured factors.")

    cluster_descs[cidx] = dict(name=cname, overview=overview, strengths=strengths, weaknesses=weaknesses)

# -----------------------------------------
# 9) Save outputs: labels, summaries, report
# -----------------------------------------
# a) Per-sample file with labels and base-dimension scores
sample_base_scores = pd.DataFrame({
    b: Fz_df[[c for c in Fz_df.columns if base_map[c] == b]].mean(axis=1) for b in base_names
})
out = df.copy()
out['cluster'] = labels_final
out['cluster_name'] = out['cluster'].map(cluster_names)
out = pd.concat([out.reset_index(drop=True), sample_base_scores.reset_index(drop=True)[base_names]], axis=1)
out.to_csv('cat_breeds_clusters.csv', index=False)

# b) Cluster centroids in z-space per individual factor
cluster_z_means.to_csv('cluster_summary_zscores.csv', index_label='cluster')

# c) Cluster centroids collapsed to base dimensions
base_summary.to_csv('cluster_summary_base_dims.csv', index_label='cluster')

# d) Human-readable text report
lines = []
lines.append(f"Optimal number of factors (>=80% variance): {n_factors}")
lines.append(f"Best number of clusters (silhouette): k={best_k}")
lines.append("\nCluster descriptions:")
for cidx in sorted(cluster_descs.keys()):
    d = cluster_descs[cidx]
    lines.append(f"Cluster {cidx} — {d['name']}")
    lines.append("  " + d['overview'])
    lines.append("  " + d['strengths'])
    lines.append("  " + d['weaknesses'])
open('cluster_report.txt', 'w', encoding='utf-8').write("\n".join(lines))

# Also print to console
print("\n".join(lines))


C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.p

Optimal number of factors (>=80% variance): 7
Best number of clusters (silhouette): k=8

Cluster descriptions:
Cluster 0 — Size & Longevity / Lower Sociability & Behavior Breeds
  This cluster comprises breeds that are balanced across factors.
  Its main strength is a balanced profile without pronounced extremes.
  No clear weaknesses emerge; trade‑offs are minimal across the measured factors.
Cluster 1 — Size & Longevity / Sociability & Behavior Breeds
  This cluster comprises breeds that are higher on Size & Longevity.
  It excels in Size & Longevity, suggesting favorable traits on these dimensions.
  No clear weaknesses emerge; trade‑offs are minimal across the measured factors.
Cluster 2 — Sociability & Behavior / Lower Size & Longevity Breeds
  This cluster comprises breeds that are higher on Sociability & Behavior and lower on Size & Longevity.
  It excels in Sociability & Behavior, suggesting favorable traits on these dimensions.
  Potential weaknesses include relatively lower s

C:\Users\amaca253\AppData\Local\Temp\ipykernel_2772\1269751964.py:124: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  base_summary = cluster_df.groupby('cluster').apply(


Cluster 0 — Lower Size & Longevity / Lower Sociability & Behavior Breeds
This cluster comprises breeds that are lower on Size & Longevity. It shows no pronounced strengths and is comparatively even elsewhere. Potential weaknesses include relatively lower scores on Size & Longevity. 

Cluster 1 — Size & Longevity / Sociability & Behavior Breeds
This cluster comprises breeds that are higher on Size & Longevity. It excels in Size & Longevity, suggesting favorable traits on these dimensions. No clear weaknesses emerge; trade‑offs are minimal across the measured factors. 

Cluster 2 — Sociability & Behavior / Lower Size & Longevity Breeds
This cluster comprises breeds that are balanced across factors. Its main strength is a balanced profile without pronounced extremes. No clear weaknesses emerge; trade‑offs are minimal across the measured factors. 

Cluster 3 — Sociability & Behavior / Lower Size & Longevity Breeds
This cluster comprises breeds that are higher on Sociability & Behavior. It excels in Sociability & Behavior, suggesting favorable traits on these dimensions. No clear weaknesses emerge; trade‑offs are minimal across the measured factors. 

Cluster 4 — Size & Longevity / Lower Sociability & Behavior Breeds
This cluster comprises breeds that are higher on Size & Longevity. It excels in Size & Longevity, suggesting favorable traits on these dimensions. No clear weaknesses emerge; trade‑offs are minimal across the measured factors. 

Cluster 5 — Lower Size & Longevity / Lower Sociability & Behavior Breeds
This cluster comprises breeds that are balanced across factors. Its main strength is a balanced profile without pronounced extremes. No clear weaknesses emerge; trade‑offs are minimal across the measured factors

In [7]:
import pandas as pd

# Load dataset
df = df

# Select the Ragdoll row (dataset label: "Ragdoll Cats")
r = df.loc[df["name"].str.strip().str.lower() == "ragdoll cats"].iloc[0]

# Extract metrics
fam = int(r["family_friendly"])
child = int(r["children_friendly"])
pets = int(r["other_pets_friendly"])
play = int(r["playfulness"])
groom = int(r["grooming"])
shed = int(r["shedding"])
health = int(r["general_health"])
min_le, max_le = float(r["min_life_expectancy"]), float(r["max_life_expectancy"])

# Build the three sentences
s1 = (
    f"Ragdoll cats excel in temperament, with family‑friendly and children‑friendly scores "
    f"of {fam}/5 and {child}/5, strong compatibility with other pets ({pets}/5), a playful "
    f"disposition ({play}/5), very low grooming needs ({groom}/5), and a solid life expectancy "
    f"of {min_le:.0f}–{max_le:.0f} years."
)
s2 = (
    f"They shed more than average ({shed}/5) and show average general health ({health}/5) "
    f"in this dataset."
)
s3 = (
    "Overall, Ragdolls are affectionate, low‑grooming companions whose higher shedding and "
    "average health are the key trade‑offs."
)

print(s1)
print(s2)
print(s3)


Ragdoll cats excel in temperament, with family‑friendly and children‑friendly scores of 5/5 and 5/5, strong compatibility with other pets (4/5), a playful disposition (4/5), very low grooming needs (1/5), and a solid life expectancy of 12–17 years.
They shed more than average (4/5) and show average general health (3/5) in this dataset.
Overall, Ragdolls are affectionate, low‑grooming companions whose higher shedding and average health are the key trade‑offs.


In [8]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Load raw data
raw = df
num = raw.select_dtypes(include=[np.number])

# Impute and scale
X = SimpleImputer(strategy='median').fit_transform(num)
X = StandardScaler().fit_transform(X)

# Select number of factors to reach >=80% variance using PCA proxy
pca = PCA().fit(X)
n_factors = int(np.argmax(np.cumsum(pca.explained_variance_ratio_) >= 0.80) + 1)

# Factor Analysis
fa = FactorAnalysis(n_components=n_factors, random_state=42)
F = fa.fit_transform(X)
loadings = pd.DataFrame(fa.components_.T, index=num.columns, columns=[f'F{i+1}' for i in range(n_factors)])

# Map factors to base groups using absolute loadings
var_groups = {
    'Size & Longevity': [c for c in num.columns if 'weight' in c or 'life_expectancy' in c],
    'Sociability & Behavior': ['family_friendly','shedding','general_health','playfulness','children_friendly','grooming','intelligence','other_pets_friendly'],
}
used = set(var_groups['Size & Longevity']) | set(var_groups['Sociability & Behavior'])
other_vars = [c for c in num.columns if c not in used]
if other_vars:
    var_groups['Other'] = other_vars

factor_to_base = {}
for f in loadings.columns:
    scores = {grp: float(np.abs(loadings.loc[var_groups[grp], f]).sum()) for grp in var_groups}
    factor_to_base[f] = max(scores, key=scores.get)

# Standardize factor scores and compute base scores per sample
Fz = StandardScaler().fit_transform(F)
Fz_df = pd.DataFrame(Fz, columns=list(loadings.columns))
base_names = sorted(set(factor_to_base.values()))
base_scores = pd.DataFrame({b: Fz_df[[f for f in Fz_df.columns if factor_to_base[f]==b]].mean(axis=1) for b in base_names})

# Pick optimal k for clustering (2..8) and cluster
best_k, best_sil = None, -1
for k in range(2,9):
    km = KMeans(n_clusters=k, n_init=50, random_state=42)
    labels = km.fit_predict(Fz_df)
    sil = silhouette_score(Fz_df, labels)
    if sil > best_sil:
        best_k, best_sil = k, sil

km_final = KMeans(n_clusters=best_k, n_init=100, random_state=42)
labels_final = km_final.fit_predict(Fz_df)

# Extract Ragdoll Cats metrics
r = raw[raw['name'].str.strip().str.lower()=='ragdoll cats'].iloc[0]
rb = base_scores.iloc[r.name]  # index aligns with raw rows

ragdoll_summary = {
    'family_friendly': int(r['family_friendly']),
    'children_friendly': int(r['children_friendly']),
    'other_pets_friendly': int(r['other_pets_friendly']),
    'grooming': int(r['grooming']),
    'shedding': int(r['shedding']),
    'general_health': int(r['general_health']),
    'playfulness': int(r['playfulness']),
    'min_weight': float(r['min_weight']),
    'max_weight': float(r['max_weight']),
    'min_life_expectancy': float(r['min_life_expectancy']),
    'max_life_expectancy': float(r['max_life_expectancy']),
    'size_longevity_z': float(rb.get('Size & Longevity', np.nan)),
    'sociability_behavior_z': float(rb.get('Sociability & Behavior', np.nan)),
}

best_k, n_factors, ragdoll_summary

C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.p

(8,
 7,
 {'family_friendly': 5,
  'children_friendly': 5,
  'other_pets_friendly': 4,
  'grooming': 1,
  'shedding': 4,
  'general_health': 3,
  'playfulness': 4,
  'min_weight': 10.0,
  'max_weight': 20.0,
  'min_life_expectancy': 12.0,
  'max_life_expectancy': 17.0,
  'size_longevity_z': 0.45634808323858395,
  'sociability_behavior_z': -0.5235847425093945})